In [0]:
# Databricks notebook source
# =========================
# Manual Config
# =========================
SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION   = "v6.1.0"
ENDPOINT         = "databricks-gpt-5-2"
RUN_SCOPE        = "PROD"

TEST_MODE        = False
TEST_MAX_GROUPS  = 20

RATE_LIMIT_SECONDS = 0.6
MAX_TOKENS         = 1600
MAX_RETRIES        = 4
BACKOFF_BASE       = 1.8

GROUP_DIMS_TO_RUN  = []      # [] 이면 전체
TARGET_Y_FEATURE   = ""   # "" 또는 None 이면 전체

PVAL_MAX           = 0.10
ABS_COEF_THRESHOLD = 0.10
USE_IS_DRIVER      = True

MAX_DRIVERS_PER_KEY_FULL    = 4
MAX_DRIVERS_PER_KEY_COMPACT = 3
MAX_DIFF_STATS_FULL         = 10

MODEL_SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_model_summary"
COEF_SUMMARY_TABLE  = "sandbox.z_jungryo_lee.voc_wls_coef_summary"

SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_summary_v61"
DETAIL_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_detail_v61"
FAILED_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_failed_v61"

print(f"[CONFIG] quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, endpoint={ENDPOINT}, run_scope={RUN_SCOPE}")
print(f"[CONFIG] dims={GROUP_DIMS_TO_RUN}, target_y_feature={TARGET_Y_FEATURE}, test_mode={TEST_MODE} (limit={TEST_MAX_GROUPS})")
print(f"[CONFIG] pval<={PVAL_MAX}, abs(coef)>={ABS_COEF_THRESHOLD}, use_is_driver={USE_IS_DRIVER}")

# =========================
# Imports
# =========================
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC
import time
import json
import traceback
import re

# =========================
# Prompt
# =========================
SYSTEM_PROMPT = """
You are a senior product strategy planner for TV products.
Return ONLY strict JSON. No preface, no markdown.

Language and style rules:
- Write all outputs in Korean.
- Every sentence must use a formal business tone ending in '~습니다', '~입니다', or equivalent formal style.
- Keep each sentence concise and intuitive for planners and strategists.
- You may use memorable strategic phrasing if it is clear and grounded in the evidence.
- Do not expose raw variable prefixes like '07_02_' or other code-like prefixes in prose.
- If you mention adjusted R-squared, keep the English term exactly as 'adjusted R-squared'.

Analysis rules:
- Base all interpretations only on the evidence provided in the payload.
- Always compare group_keys within the given group_dim.
- Do not decide which key drivers to include. The key drivers are already fixed in the payload.
- For each group_key, identify what is most distinctive relative to peers.
- Do not call a group_key average unless there is no clear distinctive positive or negative pattern in the payload.
- country_core must summarize the single most distinctive value of that group_key relative to peers.
- summary_comment must explain the comparative basis, not restate country_core.
- If a group_key has both a strong positive and a meaningful negative driver, mention both.

For each key_driver:
- Do not change x_feature.
- Do not add or remove drivers.
- Write only a short planner-friendly comment for the given driver.
- Respect the sign of coef:
  - positive: reinforcing factor
  - negative: friction or drag factor

Return ONLY strict JSON following this schema:
{
  "overall_core_message": "전체 비교 한 줄 핵심",
  "planning_summary": "기획/전략 관점 요약 멘트",
  "country_insight_cards": [
    {
      "group_key": "Brazil",
      "country_core": "브라질은 다른 국가 대비 가장 두드러지는 핵심 가치를 한 줄로 설명합니다.",
      "summary_comment": "비교 관점에서 어떤 요인이 강하고 어떤 요인이 약한지 설명합니다.",
      "key_driver_comments": [
        {
          "x_feature": "Voice Control",
          "comment": "이 driver의 기획적 의미를 설명합니다."
        }
      ],
      "special_points": [
        "다른 시장과 비교해 두드러지는 점을 설명합니다."
      ]
    }
  ],
  "strategy_takeaways": [
    "시장별로 리모컨의 핵심 가치 정의를 다르게 가져갈 필요가 있습니다."
  ]
}
""".strip()

def build_user_prompt(group_dim: str, y_feature: str) -> str:
    return f"""
You will receive one JSON payload for one y_feature.

Context:
- group_dim: {group_dim}
- y_feature: {y_feature}

Required output:
- overall_core_message
- planning_summary
- country_insight_cards:
  - group_key
  - country_core
  - summary_comment
  - key_driver_comments
  - special_points
- strategy_takeaways

Important:
- Focus on planning meaning and market distinction.
- Do not select drivers yourself.
- Only write comments for the provided fixed drivers.
- Avoid repeating the same sentence pattern across group_keys.
""".strip()

# =========================
# Helpers
# =========================
def rows_to_pylist(lst):
    if not lst:
        return []
    return [dict(x.asDict()) if hasattr(x, "asDict") else dict(x) for x in lst]

def json_dumps_safe(obj):
    return json.dumps(obj, ensure_ascii=False)

def safe_float(v, default=None):
    try:
        if v is None:
            return default
        return float(v)
    except Exception:
        return default

def safe_int(v, default=None):
    try:
        if v is None:
            return default
        return int(v)
    except Exception:
        return default

def clean_feature_name(name: str) -> str:
    if not name:
        return ""
    s = str(name)
    s = re.sub(r"^[0-9]+([_./-][0-9]+)*[_./-]*", "", s)
    s = re.sub(r"^[^A-Za-z가-힣]+", "", s)
    s = s.replace("_", " ").strip()
    return s

def to_direction(coef):
    return "positive" if safe_float(coef, 0.0) >= 0 else "negative"

def sort_drivers(drivers):
    if not drivers:
        return []
    return sorted(drivers, key=lambda d: abs(float(d.get("coef", 0.0) or 0.0)), reverse=True)

def cleanse_driver_names(drivers: list) -> list:
    out = []
    for d in drivers or []:
        x = dict(d)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        x["direction"] = x.get("direction") or to_direction(x.get("coef"))
        out.append(x)
    return out

def cleanse_comp_stats(comp_stats: list) -> list:
    out = []
    for item in comp_stats or []:
        x = dict(item)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        out.append(x)
    return out

def compute_confidence_level(model_stats: list) -> dict:
    if not model_stats:
        return {
            "level": "low",
            "score_basis": {
                "min_adj_r_squared": None,
                "max_model_p_value": None,
                "min_y_obs": None
            }
        }

    adj_r2_list = [safe_float(x.get("adj_r_squared")) for x in model_stats if safe_float(x.get("adj_r_squared")) is not None]
    pval_list   = [safe_float(x.get("model_p_value")) for x in model_stats if safe_float(x.get("model_p_value")) is not None]
    yobs_list   = [safe_int(x.get("y_obs")) for x in model_stats if safe_int(x.get("y_obs")) is not None]

    min_adj_r2 = min(adj_r2_list) if adj_r2_list else None
    max_pval   = max(pval_list) if pval_list else None
    min_y_obs  = min(yobs_list) if yobs_list else None

    if min_adj_r2 is None or max_pval is None or min_y_obs is None:
        level = "low"
    elif min_adj_r2 >= 0.60 and max_pval <= 0.05 and min_y_obs >= 100:
        level = "high"
    elif min_adj_r2 >= 0.30 and max_pval <= 0.10 and min_y_obs >= 60:
        level = "mid"
    else:
        level = "low"

    return {
        "level": level,
        "score_basis": {
            "min_adj_r_squared": min_adj_r2,
            "max_model_p_value": max_pval,
            "min_y_obs": min_y_obs
        }
    }

def confidence_band_from_level(level: str) -> str:
    if level == "high":
        return "Strong"
    if level == "mid":
        return "Moderate"
    return "Caution"

# =========================
# AI Query
# =========================
def call_ai_query(endpoint: str, request_text: str, temperature: float = 0.0, max_tokens: int = 1600):
    df = spark.createDataFrame([(request_text,)], ["p"])
    out = (
        df.select(
            F.expr(f"""
                ai_query(
                    endpoint => '{endpoint}',
                    request  => p,
                    modelParameters => named_struct(
                        'temperature', {float(temperature)},
                        'max_tokens',  {int(max_tokens)}
                    )
                )
            """).alias("json_out")
        ).first()
    )
    return out["json_out"] if out else None

def call_ai_query_with_retry(endpoint: str, system_prompt: str, user_prompt: str, payload_full_json: str, payload_compact_json: str):
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            payload_json = payload_full_json if attempt <= 2 else payload_compact_json
            req = f"[SYSTEM]\n{system_prompt}\n\n[USER]\n{user_prompt}\n\n[PAYLOAD]\n{payload_json}"
            raw = call_ai_query(endpoint, req, temperature=0.0, max_tokens=MAX_TOKENS)

            if not raw:
                raise RuntimeError("Empty response from ai_query")

            clean = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean)

            time.sleep(RATE_LIMIT_SECONDS)
            return data, clean

        except Exception as e:
            wait = BACKOFF_BASE ** (attempt - 1)
            print(f"[WARN] ai_query failed (attempt {attempt}/{MAX_RETRIES}): {repr(e)} -> wait {wait:.1f}s")
            time.sleep(wait)
            last_err = e

    raise RuntimeError(f"ai_query failed after {MAX_RETRIES} retries: {repr(last_err)}")

# =========================
# Load Source Tables
# =========================
model_summary_sdf_org = spark.table(MODEL_SUMMARY_TABLE)
coef_summary_sdf_org  = spark.table(COEF_SUMMARY_TABLE)

if GROUP_DIMS_TO_RUN:
    model_summary_sdf = model_summary_sdf_org.filter(F.col("group_dim").isin(GROUP_DIMS_TO_RUN))
    coef_summary_sdf  = coef_summary_sdf_org.filter(F.col("group_dim").isin(GROUP_DIMS_TO_RUN))
else:
    model_summary_sdf = model_summary_sdf_org
    coef_summary_sdf  = coef_summary_sdf_org

if TARGET_Y_FEATURE:
    model_summary_sdf = model_summary_sdf.filter(F.col("y_feature") == TARGET_Y_FEATURE)
    coef_summary_sdf  = coef_summary_sdf.filter(F.col("y_feature") == TARGET_Y_FEATURE)

print("[INFO] model_summary rows:", model_summary_sdf.count())
print("[INFO] coef_summary rows :", coef_summary_sdf.count())

# =========================
# Driver Filter
# =========================
drivers_sdf = coef_summary_sdf.withColumn("abs_coef", F.abs(F.col("coef")))

if USE_IS_DRIVER and "is_driver" in drivers_sdf.columns:
    drivers_sdf = drivers_sdf.filter(F.col("is_driver") == 1)

drivers_sdf = (
    drivers_sdf
    .filter(
        (F.col("p_value") <= F.lit(PVAL_MAX)) &
        (F.col("abs_coef") >= F.lit(ABS_COEF_THRESHOLD))
    )
    .cache()
)

print("[INFO] filtered driver rows:", drivers_sdf.count())

# =========================
# Aggregate
# =========================
ms_agg_sdf = (
    model_summary_sdf
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                F.col("group_key"),
                F.col("r_squared"),
                F.col("adj_r_squared"),
                F.col("f_statistic"),
                F.col("prob_f").alias("model_p_value"),
                F.col("log_likelihood"),
                F.col("aic"),
                F.col("bic"),
                F.col("y_obs"),
                F.col("cond_no")
            )
        ).alias("model_stats")
    )
)

drivers_overall_agg_sdf = (
    drivers_sdf
    .select(
        "group_dim", "y_feature", "group_key", "x_feature",
        "coef", "p_value", "t_value", "x_obs", "abs_coef"
    )
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                "group_key",
                "x_feature",
                "coef",
                "p_value",
                "t_value",
                "x_obs",
                "abs_coef"
            )
        ).alias("drivers_filtered")
    )
)

drivers_by_key_agg_sdf = (
    drivers_sdf
    .withColumn("direction", F.when(F.col("coef") >= 0, F.lit("positive")).otherwise(F.lit("negative")))
    .select(
        "group_dim", "y_feature", "group_key",
        "x_feature", "coef", "p_value", "t_value", "x_obs", "abs_coef", "direction"
    )
    .groupBy("group_dim", "y_feature", "group_key")
    .agg(
        F.collect_list(
            F.struct(
                "x_feature",
                "coef",
                "p_value",
                "t_value",
                "x_obs",
                "abs_coef",
                "direction"
            )
        ).alias("key_driver_candidates")
    )
)

payload_base_sdf = (
    ms_agg_sdf
    .join(drivers_overall_agg_sdf, ["group_dim", "y_feature"], "left")
    .cache()
)

total_groups = payload_base_sdf.count()
print("[INFO] payload base groups:", total_groups)

if TEST_MODE and total_groups > TEST_MAX_GROUPS:
    payload_base_sdf = payload_base_sdf.limit(TEST_MAX_GROUPS).cache()
    print(f"[INFO] TEST_MODE -> limited to {TEST_MAX_GROUPS}")

# =========================
# Comparative Driver Stats
# =========================
coef_for_compare = (
    coef_summary_sdf
    .select("group_dim", "group_key", "y_feature", "x_feature", "coef")
    .cache()
)

spread_stats_sdf = (
    coef_for_compare
    .groupBy("group_dim", "y_feature", "x_feature")
    .agg(
        F.min("coef").alias("coef_min"),
        F.max("coef").alias("coef_max"),
        F.avg("coef").alias("coef_mean"),
        F.stddev("coef").alias("coef_std")
    )
    .withColumn("coef_spread", F.col("coef_max") - F.col("coef_min"))
)

by_key_list_sdf = (
    coef_for_compare
    .groupBy("group_dim", "y_feature", "x_feature")
    .agg(
        F.collect_list(
            F.struct(F.col("group_key"), F.col("coef"))
        ).alias("by_key_coef_list")
    )
)

comparative_agg_sdf = (
    spread_stats_sdf
    .join(by_key_list_sdf, ["group_dim", "y_feature", "x_feature"], "left")
    .groupBy("group_dim", "y_feature")
    .agg(
        F.collect_list(
            F.struct(
                "x_feature",
                "coef_min",
                "coef_max",
                "coef_mean",
                "coef_std",
                "coef_spread",
                "by_key_coef_list"
            )
        ).alias("x_feature_diff_stats")
    )
)

comp_rows = {(r["group_dim"], r["y_feature"]): r for r in comparative_agg_sdf.collect()}

# key-level driver candidates dictionary
key_driver_rows = drivers_by_key_agg_sdf.collect()
key_driver_map = {
    (r["group_dim"], r["y_feature"], r["group_key"]): cleanse_driver_names(
        sort_drivers(rows_to_pylist(r["key_driver_candidates"]))
    )
    for r in key_driver_rows
}

# =========================
# Payload + Mapping
# =========================
def build_payload(base_row, comp_rows_dict, key_driver_map, compact=False):
    group_dim = base_row["group_dim"]
    y_feature = base_row["y_feature"]

    model_summary = rows_to_pylist(base_row["model_stats"])
    confidence_meta = compute_confidence_level(model_summary)

    drivers = rows_to_pylist(base_row["drivers_filtered"]) if base_row["drivers_filtered"] else []
    drivers = cleanse_driver_names(sort_drivers(drivers))
    drivers = drivers[:MAX_DRIVERS_PER_KEY_COMPACT] if compact else drivers[:12]

    group_keys = sorted(list({x.get("group_key") for x in model_summary if x.get("group_key") is not None}))

    key_driver_candidates = []
    for group_key in group_keys:
        candidates = key_driver_map.get((group_dim, y_feature, group_key), [])
        candidates = candidates[:MAX_DRIVERS_PER_KEY_COMPACT] if compact else candidates[:MAX_DRIVERS_PER_KEY_FULL]
        key_driver_candidates.append({
            "group_key": group_key,
            "drivers": [
                {
                    "x_feature": d.get("x_feature"),
                    "coef": d.get("coef"),
                    "p_value": d.get("p_value"),
                    "direction": d.get("direction")
                }
                for d in candidates
            ]
        })

    payload = {
        "meta": {
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "endpoint": ENDPOINT,
            "run_scope": RUN_SCOPE,
            "generated_at": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "driver_filter": {
                "p_value_max": PVAL_MAX,
                "abs_coef_threshold": ABS_COEF_THRESHOLD,
                "use_is_driver": USE_IS_DRIVER
            },
            "confidence_rule_basis": confidence_meta["score_basis"]
        },
        "context": {
            "group_dim": group_dim,
            "y_feature_raw": y_feature,
            "y_feature": clean_feature_name(y_feature)
        },
        "model_summary": model_summary,
        "drivers_filtered": drivers,
        "key_driver_candidates": key_driver_candidates
    }

    if not compact:
        comp = comp_rows_dict.get((group_dim, y_feature))
        comp_stats = cleanse_comp_stats(rows_to_pylist(comp["x_feature_diff_stats"])) if comp else []
        comp_stats = sorted(comp_stats, key=lambda x: abs(float(x.get("coef_spread", 0.0) or 0.0)), reverse=True)[:MAX_DIFF_STATS_FULL]
        payload["x_feature_diff_stats"] = comp_stats

    return payload

def llm_json_to_summary_record(group_dim, y_feature, llm_json, payload_str, computed_confidence):
    return {
        "snapshot_quarter": SNAPSHOT_QUARTER,
        "prompt_version": PROMPT_VERSION,
        "group_dim": group_dim,
        "y_feature": y_feature,
        "y_feature_clean": clean_feature_name(y_feature),
        "overall_core_message": llm_json.get("overall_core_message", ""),
        "planning_summary": llm_json.get("planning_summary", ""),
        "strategy_takeaways_json": json_dumps_safe(llm_json.get("strategy_takeaways", [])),
        "confidence_level": computed_confidence.get("level", "low"),
        "confidence_band": confidence_band_from_level(computed_confidence.get("level", "low")),
        "raw_llm_json": json_dumps_safe(llm_json),
        "payload_json": payload_str,
        "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
    }

def llm_json_to_detail_records(group_dim, y_feature, llm_json, key_driver_map):
    detail_records = []
    cards = llm_json.get("country_insight_cards", []) or []

    for card in cards:
        group_key = card.get("group_key", "")
        fixed_drivers = key_driver_map.get((group_dim, y_feature, group_key), [])
        llm_comments = {
            x.get("x_feature"): x.get("comment", "")
            for x in (card.get("key_driver_comments", []) or [])
            if x.get("x_feature")
        }

        key_drivers_final = []
        for d in fixed_drivers:
            key_drivers_final.append({
                "x_feature": d.get("x_feature"),
                "coef": d.get("coef"),
                "p_value": d.get("p_value"),
                "direction": d.get("direction"),
                "comment": llm_comments.get(d.get("x_feature"), "")
            })

        detail_records.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "group_key": group_key,
            "y_feature": y_feature,
            "y_feature_clean": clean_feature_name(y_feature),
            "country_core": card.get("country_core", ""),
            "summary_comment": card.get("summary_comment", ""),
            "key_drivers_json": json_dumps_safe(key_drivers_final),
            "special_points_json": json_dumps_safe(card.get("special_points", [])),
            "raw_card_json": json_dumps_safe(card),
            "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
        })

    return detail_records

In [0]:
# =========================
# Generate
# =========================
base_rows = payload_base_sdf.collect()
print(f"[INFO] generation targets: {len(base_rows)}")

summary_results = []
detail_results  = []
failed          = []

for i, r in enumerate(base_rows, 1):
    group_dim = r["group_dim"]
    y_feature = r["y_feature"]

    try:
        payload_full = build_payload(r, comp_rows, key_driver_map, compact=False)
        payload_compact = build_payload(r, comp_rows, key_driver_map, compact=True)

        payload_full_str = json.dumps(payload_full, ensure_ascii=False)
        payload_compact_str = json.dumps(payload_compact, ensure_ascii=False)

        computed_confidence = compute_confidence_level(payload_full["model_summary"])

        user_prompt = build_user_prompt(
            group_dim=group_dim,
            y_feature=clean_feature_name(y_feature)
        )

        llm_json, raw = call_ai_query_with_retry(
            endpoint=ENDPOINT,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=user_prompt,
            payload_full_json=payload_full_str,
            payload_compact_json=payload_compact_str
        )

        summary_results.append(
            llm_json_to_summary_record(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                payload_str=payload_compact_str,
                computed_confidence=computed_confidence
            )
        )

        detail_results.extend(
            llm_json_to_detail_records(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                key_driver_map=key_driver_map
            )
        )

        if i % 5 == 0 or i == len(base_rows):
            print(f"[INFO] progress {i}/{len(base_rows)}")

    except Exception as e:
        print(f"[ERROR] ({group_dim}, {y_feature}) failed: {repr(e)}")
        traceback.print_exc()

        failed.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "y_feature": y_feature,
            "error": repr(e),
            "payload_json": json.dumps(
                {"context": {"group_dim": group_dim, "y_feature": y_feature}},
                ensure_ascii=False
            ),
            "failed_at": datetime.now(UTC)
        })

print(f"[INFO] summary success={len(summary_results)}, detail success={len(detail_results)}, failed={len(failed)}")

# =========================
# Save Local
# =========================
with open("results_summary_v61.json", "w", encoding="utf-8") as f:
    json.dump(summary_results, f, ensure_ascii=False, indent=2)

with open("results_detail_v61.json", "w", encoding="utf-8") as f:
    json.dump(detail_results, f, ensure_ascii=False, indent=2)

print("[INFO] saved local files: results_summary_v61.json, results_detail_v61.json")

# =========================
# Create Tables
# =========================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SUMMARY_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
y_feature STRING,
y_feature_clean STRING,
overall_core_message STRING,
planning_summary STRING,
strategy_takeaways_json STRING,
confidence_level STRING,
confidence_band STRING,
raw_llm_json STRING,
payload_json STRING,
generated_at TIMESTAMP
) USING delta
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DETAIL_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
group_key STRING,
y_feature STRING,
y_feature_clean STRING,
country_core STRING,
summary_comment STRING,
key_drivers_json STRING,
special_points_json STRING,
raw_card_json STRING,
generated_at TIMESTAMP
) USING delta
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FAILED_TABLE} (
snapshot_quarter STRING,
prompt_version STRING,
group_dim STRING,
y_feature STRING,
error STRING,
payload_json STRING,
failed_at TIMESTAMP
) USING delta
""")

# =========================
# Merge Summary
# =========================
if summary_results:
    summary_sdf = spark.createDataFrame([Row(**rec) for rec in summary_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("overall_core_message").cast("string").alias("overall_core_message"),
        F.col("planning_summary").cast("string").alias("planning_summary"),
        F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
        F.col("confidence_level").cast("string").alias("confidence_level"),
        F.col("confidence_band").cast("string").alias("confidence_band"),
        F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    summary_sdf.createOrReplaceTempView("insight_summary_updates_v61")

    spark.sql(f"""
        MERGE INTO {SUMMARY_TABLE} AS t
        USING insight_summary_updates_v61 AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
        t.y_feature_clean         = s.y_feature_clean,
        t.overall_core_message    = s.overall_core_message,
        t.planning_summary        = s.planning_summary,
        t.strategy_takeaways_json = s.strategy_takeaways_json,
        t.confidence_level        = s.confidence_level,
        t.confidence_band         = s.confidence_band,
        t.raw_llm_json            = s.raw_llm_json,
        t.payload_json            = s.payload_json,
        t.generated_at            = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
        snapshot_quarter,
        prompt_version,
        group_dim,
        y_feature,
        y_feature_clean,
        overall_core_message,
        planning_summary,
        strategy_takeaways_json,
        confidence_level,
        confidence_band,
        raw_llm_json,
        payload_json,
        generated_at
        )
        VALUES (
        s.snapshot_quarter,
        s.prompt_version,
        s.group_dim,
        s.y_feature,
        s.y_feature_clean,
        s.overall_core_message,
        s.planning_summary,
        s.strategy_takeaways_json,
        s.confidence_level,
        s.confidence_band,
        s.raw_llm_json,
        s.payload_json,
        s.generated_at
        )
    """)

    print(f"[INFO] merged summary rows: {summary_sdf.count()}")

# =========================
# Merge Detail
# =========================
if detail_results:
    detail_sdf = spark.createDataFrame([Row(**rec) for rec in detail_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("group_key").cast("string").alias("group_key"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("country_core").cast("string").alias("country_core"),
        F.col("summary_comment").cast("string").alias("summary_comment"),
        F.col("key_drivers_json").cast("string").alias("key_drivers_json"),
        F.col("special_points_json").cast("string").alias("special_points_json"),
        F.col("raw_card_json").cast("string").alias("raw_card_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    detail_sdf.createOrReplaceTempView("insight_detail_updates_v61")

    spark.sql(f"""
        MERGE INTO {DETAIL_TABLE} AS t
        USING insight_detail_updates_v61 AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.group_key        = s.group_key
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
        t.y_feature_clean     = s.y_feature_clean,
        t.country_core        = s.country_core,
        t.summary_comment     = s.summary_comment,
        t.key_drivers_json    = s.key_drivers_json,
        t.special_points_json = s.special_points_json,
        t.raw_card_json       = s.raw_card_json,
        t.generated_at        = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
        snapshot_quarter,
        prompt_version,
        group_dim,
        group_key,
        y_feature,
        y_feature_clean,
        country_core,
        summary_comment,
        key_drivers_json,
        special_points_json,
        raw_card_json,
        generated_at
        )
        VALUES (
        s.snapshot_quarter,
        s.prompt_version,
        s.group_dim,
        s.group_key,
        s.y_feature,
        s.y_feature_clean,
        s.country_core,
        s.summary_comment,
        s.key_drivers_json,
        s.special_points_json,
        s.raw_card_json,
        s.generated_at
        )
    """)

    print(f"[INFO] merged detail rows: {detail_sdf.count()}")

# =========================
# Save Failed
# =========================
if failed:
    failed_sdf = spark.createDataFrame([Row(**rec) for rec in failed]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("error").cast("string").alias("error"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.col("failed_at").cast("timestamp").alias("failed_at")
    )
    failed_sdf.write.mode("append").saveAsTable(FAILED_TABLE)
    print(f"[WARN] failed logged: {failed_sdf.count()}")

# =========================
# Display
# =========================
print("[INFO] Summary table preview")
display(
    spark.table(SUMMARY_TABLE)
        .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
        .filter(F.col("prompt_version") == PROMPT_VERSION)
        .orderBy(F.desc("generated_at"))
)

print("[INFO] Detail table preview")
display(
    spark.table(DETAIL_TABLE)
        .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
        .filter(F.col("prompt_version") == PROMPT_VERSION)
        .orderBy(F.col("group_dim"), F.col("y_feature"), F.col("group_key"))
)


In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC
import json
import traceback

MAX_TOKENS = 800
MAX_DRIVERS_PER_KEY_FULL = 2
MAX_DRIVERS_PER_KEY_COMPACT = 1
MAX_DIFF_STATS_FULL = 0



# 1) 실패 대상 조회
retry_failed_sdf = (
    spark.table(FAILED_TABLE)
    .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
    .filter(F.col("prompt_version") == PROMPT_VERSION)
    .select("snapshot_quarter", "prompt_version", "group_dim", "y_feature", "error", "failed_at")
    .dropDuplicates(["snapshot_quarter", "prompt_version", "group_dim", "y_feature"])
)

# 이미 성공 적재된 summary는 제외
retry_targets_sdf = (
    retry_failed_sdf.alias("f")
    .join(
        spark.table(SUMMARY_TABLE)
            .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
            .filter(F.col("prompt_version") == PROMPT_VERSION)
            .select("group_dim", "y_feature")
            .dropDuplicates()
            .alias("s"),
        on=[
            F.col("f.group_dim") == F.col("s.group_dim"),
            F.col("f.y_feature") == F.col("s.y_feature"),
        ],
        how="left_anti"
    )
    .select(F.col("f.group_dim").alias("group_dim"), F.col("f.y_feature").alias("y_feature"))
    .dropDuplicates()
)

retry_keys = {
    (r["group_dim"], r["y_feature"])
    for r in retry_targets_sdf.collect()
}

base_rows_retry = [
    r for r in payload_base_sdf.collect()
    if (r["group_dim"], r["y_feature"]) in retry_keys
]

print(f"[INFO] retry targets: {len(base_rows_retry)}")

# 2) 재실행
retry_summary_results = []
retry_detail_results = []
retry_failed = []

for i, r in enumerate(base_rows_retry, 1):
    group_dim = r["group_dim"]
    y_feature = r["y_feature"]

    try:
        payload_full = build_payload(r, comp_rows, key_driver_map, compact=False)
        payload_compact = build_payload(r, comp_rows, key_driver_map, compact=True)

        payload_full_str = json.dumps(payload_full, ensure_ascii=False)
        payload_compact_str = json.dumps(payload_compact, ensure_ascii=False)

        computed_confidence = compute_confidence_level(payload_full["model_summary"])

        user_prompt = build_user_prompt(
            group_dim=group_dim,
            y_feature=clean_feature_name(y_feature)
        )

        llm_json, raw = call_ai_query_with_retry(
            endpoint=ENDPOINT,
            system_prompt=SYSTEM_PROMPT,
            user_prompt=user_prompt,
            payload_full_json=payload_full_str,
            payload_compact_json=payload_compact_str
        )

        retry_summary_results.append(
            llm_json_to_summary_record(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                payload_str=payload_compact_str,
                computed_confidence=computed_confidence
            )
        )

        retry_detail_results.extend(
            llm_json_to_detail_records(
                group_dim=group_dim,
                y_feature=y_feature,
                llm_json=llm_json,
                key_driver_map=key_driver_map
            )
        )

        if i % 5 == 0 or i == len(base_rows_retry):
            print(f"[INFO] retry progress {i}/{len(base_rows_retry)}")

    except Exception as e:
        print(f"[ERROR] retry failed ({group_dim}, {y_feature}): {repr(e)}")
        traceback.print_exc()

        retry_failed.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "y_feature": y_feature,
            "error": repr(e),
            "payload_json": json.dumps(
                {"context": {"group_dim": group_dim, "y_feature": y_feature}},
                ensure_ascii=False
            ),
            "failed_at": datetime.now(UTC)
        })

print(f"[INFO] retry result: summary={len(retry_summary_results)}, detail={len(retry_detail_results)}, failed={len(retry_failed)}")

# 3) summary 머지
if retry_summary_results:
    retry_summary_sdf = spark.createDataFrame([Row(**rec) for rec in retry_summary_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("overall_core_message").cast("string").alias("overall_core_message"),
        F.col("planning_summary").cast("string").alias("planning_summary"),
        F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
        F.col("confidence_level").cast("string").alias("confidence_level"),
        F.col("confidence_band").cast("string").alias("confidence_band"),
        F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    retry_summary_sdf.createOrReplaceTempView("insight_summary_retry_v61")

    spark.sql(f"""
        MERGE INTO {SUMMARY_TABLE} AS t
        USING insight_summary_retry_v61 AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean         = s.y_feature_clean,
          t.overall_core_message    = s.overall_core_message,
          t.planning_summary        = s.planning_summary,
          t.strategy_takeaways_json = s.strategy_takeaways_json,
          t.confidence_level        = s.confidence_level,
          t.confidence_band         = s.confidence_band,
          t.raw_llm_json            = s.raw_llm_json,
          t.payload_json            = s.payload_json,
          t.generated_at            = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter,
          prompt_version,
          group_dim,
          y_feature,
          y_feature_clean,
          overall_core_message,
          planning_summary,
          strategy_takeaways_json,
          confidence_level,
          confidence_band,
          raw_llm_json,
          payload_json,
          generated_at
        )
        VALUES (
          s.snapshot_quarter,
          s.prompt_version,
          s.group_dim,
          s.y_feature,
          s.y_feature_clean,
          s.overall_core_message,
          s.planning_summary,
          s.strategy_takeaways_json,
          s.confidence_level,
          s.confidence_band,
          s.raw_llm_json,
          s.payload_json,
          s.generated_at
        )
    """)

    print(f"[INFO] retried summary merged: {retry_summary_sdf.count()}")
else:
    print("[INFO] no retry summary rows")

# 4) detail 머지
if retry_detail_results:
    retry_detail_sdf = spark.createDataFrame([Row(**rec) for rec in retry_detail_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("group_key").cast("string").alias("group_key"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("country_core").cast("string").alias("country_core"),
        F.col("summary_comment").cast("string").alias("summary_comment"),
        F.col("key_drivers_json").cast("string").alias("key_drivers_json"),
        F.col("special_points_json").cast("string").alias("special_points_json"),
        F.col("raw_card_json").cast("string").alias("raw_card_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    retry_detail_sdf.createOrReplaceTempView("insight_detail_retry_v61")

    spark.sql(f"""
        MERGE INTO {DETAIL_TABLE} AS t
        USING insight_detail_retry_v61 AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.group_key        = s.group_key
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean     = s.y_feature_clean,
          t.country_core        = s.country_core,
          t.summary_comment     = s.summary_comment,
          t.key_drivers_json    = s.key_drivers_json,
          t.special_points_json = s.special_points_json,
          t.raw_card_json       = s.raw_card_json,
          t.generated_at        = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter,
          prompt_version,
          group_dim,
          group_key,
          y_feature,
          y_feature_clean,
          country_core,
          summary_comment,
          key_drivers_json,
          special_points_json,
          raw_card_json,
          generated_at
        )
        VALUES (
          s.snapshot_quarter,
          s.prompt_version,
          s.group_dim,
          s.group_key,
          s.y_feature,
          s.y_feature_clean,
          s.country_core,
          s.summary_comment,
          s.key_drivers_json,
          s.special_points_json,
          s.raw_card_json,
          s.generated_at
        )
    """)

    print(f"[INFO] retried detail merged: {retry_detail_sdf.count()}")
else:
    print("[INFO] no retry detail rows")

# 5) 재실패 적재
if retry_failed:
    retry_failed_sdf = spark.createDataFrame([Row(**rec) for rec in retry_failed]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("error").cast("string").alias("error"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.col("failed_at").cast("timestamp").alias("failed_at")
    )
    retry_failed_sdf.write.mode("append").saveAsTable(FAILED_TABLE)
    print(f"[WARN] retry failed logged: {retry_failed_sdf.count()}")

# 6) 성공한 실패건 자동 삭제
successful_retry_keys = [
    (rec["group_dim"], rec["y_feature"])
    for rec in retry_summary_results
]

def sql_escape(s):
    return str(s).replace("'", "''")

successful_retry_keys = [
    (rec["group_dim"], rec["y_feature"])
    for rec in retry_summary_results
]

if successful_retry_keys:
    delete_condition = " OR ".join([
        f"(group_dim = '{sql_escape(g)}' AND y_feature = '{sql_escape(y)}')"
        for g, y in successful_retry_keys
    ])

    spark.sql(f"""
        DELETE FROM {FAILED_TABLE}
        WHERE snapshot_quarter = '{sql_escape(SNAPSHOT_QUARTER)}'
          AND prompt_version = '{sql_escape(PROMPT_VERSION)}'
          AND ({delete_condition})
    """)

    print(f"[INFO] removed successful retry keys from failed table: {len(successful_retry_keys)}")
else:
    print("[INFO] no successful retry keys to delete")

# 7) 확인
print("[INFO] Remaining failed rows")
display(
    spark.table(FAILED_TABLE)
        .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
        .filter(F.col("prompt_version") == PROMPT_VERSION)
        .orderBy(F.desc("failed_at"))
)

print("[INFO] Summary preview")
display(
    spark.table(SUMMARY_TABLE)
        .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
        .filter(F.col("prompt_version") == PROMPT_VERSION)
        .orderBy(F.desc("generated_at"))
)

spark.sql(f"""
DELETE FROM {FAILED_TABLE}
WHERE (snapshot_quarter, prompt_version, group_dim, y_feature, failed_at) IN (
  SELECT snapshot_quarter, prompt_version, group_dim, y_feature, failed_at
  FROM (
    SELECT
      snapshot_quarter,
      prompt_version,
      group_dim,
      y_feature,
      failed_at,
      ROW_NUMBER() OVER (
        PARTITION BY snapshot_quarter, prompt_version, group_dim, y_feature
        ORDER BY failed_at DESC
      ) AS rn
    FROM {FAILED_TABLE}
    WHERE snapshot_quarter = '{SNAPSHOT_QUARTER}'
      AND prompt_version = '{PROMPT_VERSION}'
  ) t
  WHERE rn > 1
)
""")

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC
import json
import time
import traceback
import re

# ===== config =====
SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION   = "v6.1.0"

MODEL_SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_model_summary"
COEF_SUMMARY_TABLE  = "sandbox.z_jungryo_lee.voc_wls_coef_summary"

SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_summary_v61"
DETAIL_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_detail_v61"
FAILED_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_failed_v61"

ENDPOINT = "databricks-gpt-5-2"

PVAL_MAX = 0.10
ABS_COEF_THRESHOLD = 0.10
USE_IS_DRIVER = True
MAX_DRIVERS_PER_KEY = 3

MAX_TOKENS_DETAIL = 900
MAX_RETRIES = 4
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.6

# ===== prompts =====
SYSTEM_PROMPT_DETAIL = """
You are a senior product strategy planner for TV products.
Return ONLY strict JSON. No preface, no markdown.

Language rules:
- Write all outputs in Korean.
- Every sentence must use formal business tone such as '~습니다' or '~입니다'.
- Keep the writing concise and intuitive for planners.
- Do not expose raw variable prefixes like '07_02_' in prose.

Rules:
- Use only the payload evidence.
- This task is for one group_key only.
- Do not choose drivers yourself.
- Use the provided fixed drivers only.
- country_core must summarize the most distinctive value of this group_key.
- summary_comment must explain the strongest positive factor and, if present, the strongest negative factor.
- Do not describe the group_key as average if there is any meaningful positive or negative driver.
- For each driver, write only a short planner-friendly comment.

Return ONLY strict JSON in this schema:
{
  "group_key": "Brazil",
  "country_core": "핵심 한 줄",
  "summary_comment": "비교 설명",
  "key_driver_comments": [
    {
      "x_feature": "Voice Control",
      "comment": "기획적 의미"
    }
  ],
  "special_points": [
    "특이사항"
  ]
}
""".strip()

def build_user_prompt_detail(group_dim: str, group_key: str, y_feature: str) -> str:
    return f"""
You will receive one JSON payload for one group_key insight.

Context:
- group_dim: {group_dim}
- group_key: {group_key}
- y_feature: {y_feature}

Required output:
- group_key
- country_core
- summary_comment
- key_driver_comments
- special_points
""".strip()

# ===== helpers =====
def sql_escape(s):
    return str(s).replace("'", "''")

def rows_to_pylist(lst):
    if not lst:
        return []
    return [dict(x.asDict()) if hasattr(x, "asDict") else dict(x) for x in lst]

def safe_float(v, default=None):
    try:
        if v is None:
            return default
        return float(v)
    except Exception:
        return default

def safe_int(v, default=None):
    try:
        if v is None:
            return default
        return int(v)
    except Exception:
        return default

def clean_feature_name(name: str) -> str:
    if not name:
        return ""
    s = str(name)
    s = re.sub(r"^[0-9]+([_./-][0-9]+)*[_./-]*", "", s)
    s = re.sub(r"^[^A-Za-z가-힣]+", "", s)
    return s.replace("_", " ").strip()

def to_direction(coef):
    return "positive" if safe_float(coef, 0.0) >= 0 else "negative"

def sort_drivers(drivers):
    if not drivers:
        return []
    return sorted(drivers, key=lambda d: abs(float(d.get("coef", 0.0) or 0.0)), reverse=True)

def cleanse_driver_names(drivers):
    out = []
    for d in drivers or []:
        x = dict(d)
        x["x_feature"] = clean_feature_name(x.get("x_feature"))
        x["direction"] = x.get("direction") or to_direction(x.get("coef"))
        out.append(x)
    return out

def compute_confidence_level(model_stats: list) -> dict:
    if not model_stats:
        return {"level": "low", "score_basis": {"min_adj_r_squared": None, "max_model_p_value": None, "min_y_obs": None}}

    adj_r2_list = [safe_float(x.get("adj_r_squared")) for x in model_stats if safe_float(x.get("adj_r_squared")) is not None]
    pval_list   = [safe_float(x.get("model_p_value")) for x in model_stats if safe_float(x.get("model_p_value")) is not None]
    yobs_list   = [safe_int(x.get("y_obs")) for x in model_stats if safe_int(x.get("y_obs")) is not None]

    min_adj_r2 = min(adj_r2_list) if adj_r2_list else None
    max_pval   = max(pval_list) if pval_list else None
    min_y_obs  = min(yobs_list) if yobs_list else None

    if min_adj_r2 is None or max_pval is None or min_y_obs is None:
        level = "low"
    elif min_adj_r2 >= 0.60 and max_pval <= 0.05 and min_y_obs >= 100:
        level = "high"
    elif min_adj_r2 >= 0.30 and max_pval <= 0.10 and min_y_obs >= 60:
        level = "mid"
    else:
        level = "low"

    return {
        "level": level,
        "score_basis": {
            "min_adj_r_squared": min_adj_r2,
            "max_model_p_value": max_pval,
            "min_y_obs": min_y_obs
        }
    }

def confidence_band_from_level(level: str) -> str:
    return "Strong" if level == "high" else "Moderate" if level == "mid" else "Caution"

def split_pos_neg(drivers):
    pos = [d for d in (drivers or []) if d.get("direction") == "positive"]
    neg = [d for d in (drivers or []) if d.get("direction") == "negative"]
    pos = sorted(pos, key=lambda x: abs(float(x.get("coef", 0.0) or 0.0)), reverse=True)
    neg = sorted(neg, key=lambda x: abs(float(x.get("coef", 0.0) or 0.0)), reverse=True)
    return pos, neg

def make_overall_core_message(y_feature_clean, detail_seed_rows):
    strongest = []
    for row in detail_seed_rows:
        pos, _ = split_pos_neg(row.get("drivers", []))
        if pos:
            strongest.append((row["group_key"], pos[0]["x_feature"], abs(float(pos[0].get("coef", 0.0) or 0.0))))
    if strongest:
        strongest = sorted(strongest, key=lambda x: x[2], reverse=True)
        g, xf, _ = strongest[0]
        return f"{y_feature_clean}는 group_key별로 핵심 driver가 다르게 나타나며, {g}에서는 {xf}가 가장 두드러진 축으로 확인됩니다."
    return f"{y_feature_clean}는 group_key별 뚜렷한 driver 차이가 제한적으로 나타나 보수적 해석이 필요한 구조입니다."

def make_planning_summary(y_feature_clean, detail_seed_rows):
    points = []
    for row in detail_seed_rows:
        pos, _ = split_pos_neg(row.get("drivers", []))
        if pos:
            points.append(f"{row['group_key']}의 {pos[0]['x_feature']}")
    if points:
        return f"{y_feature_clean}는 group_key별 강하게 작용하는 핵심 요인이 서로 다르게 나타납니다. 대표적으로 {', '.join(points[:3])}가 핵심 축으로 확인되어 공통 접근보다 group_key별 차별화 기획이 적합합니다."
    return f"{y_feature_clean}는 유의한 driver 수가 제한적이어서 특정 기능 축보다 전반 경험 안정성 중심으로 해석하는 편이 적절합니다."

def make_strategy_takeaways(detail_seed_rows):
    takeaways = []
    positives, negatives = [], []

    for row in detail_seed_rows:
        pos, neg = split_pos_neg(row.get("drivers", []))
        if pos:
            positives.append((row["group_key"], pos[0]))
        if neg:
            negatives.append((row["group_key"], neg[0]))

    if positives:
        g, d = sorted(positives, key=lambda x: abs(float(x[1].get("coef", 0.0) or 0.0)), reverse=True)[0]
        takeaways.append(f"{g}는 {d['x_feature']} 중심의 강점 강화가 우선 과제입니다.")
    if negatives:
        g, d = sorted(negatives, key=lambda x: abs(float(x[1].get("coef", 0.0) or 0.0)), reverse=True)[0]
        takeaways.append(f"{g}는 {d['x_feature']} 관련 마찰을 우선 완화할 필요가 있습니다.")
    if len(detail_seed_rows) >= 2:
        takeaways.append("group_key별 핵심 가치 축이 다르므로 단일 메시지보다 세분화된 기획 전략이 적합합니다.")
    if not takeaways:
        takeaways.append("유의한 driver가 제한적이므로 보수적으로 해석할 필요가 있습니다.")
    return takeaways[:3]

def call_ai_query(endpoint: str, request_text: str, temperature: float = 0.0, max_tokens: int = 900):
    df = spark.createDataFrame([(request_text,)], ["p"])
    out = (
        df.select(
            F.expr(f"""
                ai_query(
                    endpoint => '{endpoint}',
                    request  => p,
                    modelParameters => named_struct(
                        'temperature', {float(temperature)},
                        'max_tokens',  {int(max_tokens)}
                    )
                )
            """).alias("json_out")
        ).first()
    )
    return out["json_out"] if out else None

def call_ai_query_with_retry_detail(endpoint, system_prompt, user_prompt, payload_json):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            req = f"[SYSTEM]\n{system_prompt}\n\n[USER]\n{user_prompt}\n\n[PAYLOAD]\n{payload_json}"
            raw = call_ai_query(endpoint, req, temperature=0.0, max_tokens=MAX_TOKENS_DETAIL)
            if not raw:
                raise RuntimeError("Empty response from ai_query")
            clean = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean)
            time.sleep(RATE_LIMIT_SECONDS)
            return data, clean
        except Exception as e:
            wait = BACKOFF_BASE ** (attempt - 1)
            print(f"[WARN] detail retry failed (attempt {attempt}/{MAX_RETRIES}): {repr(e)} -> wait {wait:.1f}s")
            time.sleep(wait)
            last_err = e
    raise RuntimeError(f"detail retry failed after {MAX_RETRIES} retries: {repr(last_err)}")

# 1. failed pair 읽기
failed_pairs = (
    spark.table(FAILED_TABLE)
    .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
    .filter(F.col("prompt_version") == PROMPT_VERSION)
    .select("group_dim", "y_feature")
    .dropDuplicates()
    .collect()
)

failed_pair_set = {(r["group_dim"], r["y_feature"]) for r in failed_pairs}
print(f"[INFO] failed pairs: {len(failed_pair_set)}")

if not failed_pair_set:
    print("[INFO] no failed pairs")
else:
    # 2. 원천테이블에서 실패 pair만 재구성
    model_summary_sdf = spark.table(MODEL_SUMMARY_TABLE)
    coef_summary_sdf  = spark.table(COEF_SUMMARY_TABLE)

    pair_filter = None
    for g, y in failed_pair_set:
        cond = (F.col("group_dim") == g) & (F.col("y_feature") == y)
        pair_filter = cond if pair_filter is None else (pair_filter | cond)

    model_summary_sdf = model_summary_sdf.filter(pair_filter)
    coef_summary_sdf  = coef_summary_sdf.filter(pair_filter)

    drivers_sdf = coef_summary_sdf.withColumn("abs_coef", F.abs(F.col("coef")))
    if USE_IS_DRIVER and "is_driver" in drivers_sdf.columns:
        drivers_sdf = drivers_sdf.filter(F.col("is_driver") == 1)

    drivers_sdf = drivers_sdf.filter(
        (F.col("p_value") <= F.lit(PVAL_MAX)) &
        (F.col("abs_coef") >= F.lit(ABS_COEF_THRESHOLD))
    )

    ms_by_key_rows = model_summary_sdf.select(
        "group_dim", "y_feature", "group_key", "r_squared", "adj_r_squared",
        F.col("prob_f").alias("model_p_value"), "y_obs"
    ).collect()

    model_summary_map = {
        (r["group_dim"], r["y_feature"], r["group_key"]): {
            "group_key": r["group_key"],
            "r_squared": r["r_squared"],
            "adj_r_squared": r["adj_r_squared"],
            "model_p_value": r["model_p_value"],
            "y_obs": r["y_obs"]
        }
        for r in ms_by_key_rows
    }

    key_driver_rows = (
        drivers_sdf
        .withColumn("direction", F.when(F.col("coef") >= 0, F.lit("positive")).otherwise(F.lit("negative")))
        .select("group_dim", "y_feature", "group_key", "x_feature", "coef", "p_value", "direction")
        .collect()
    )

    key_driver_map = {}
    for r in key_driver_rows:
        k = (r["group_dim"], r["y_feature"], r["group_key"])
        key_driver_map.setdefault(k, []).append({
            "x_feature": clean_feature_name(r["x_feature"]),
            "coef": r["coef"],
            "p_value": r["p_value"],
            "direction": r["direction"]
        })

    for k in list(key_driver_map.keys()):
        key_driver_map[k] = sort_drivers(key_driver_map[k])[:DETAIL_MAX_DRIVERS_PER_KEY]

    pair_model_stats = (
        model_summary_sdf
        .groupBy("group_dim", "y_feature")
        .agg(
            F.collect_list(
                F.struct(
                    "group_key",
                    "r_squared",
                    "adj_r_squared",
                    F.col("prob_f").alias("model_p_value"),
                    "y_obs"
                )
            ).alias("model_stats")
        )
        .collect()
    )
    pair_model_stats_map = {(r["group_dim"], r["y_feature"]): rows_to_pylist(r["model_stats"]) for r in pair_model_stats}

    # 3. detail 생성
    detail_results = []
    failed_detail = []

    for group_dim, y_feature in failed_pair_set:
        group_keys = sorted([k[2] for k in key_driver_map.keys() if k[0] == group_dim and k[1] == y_feature])
        if not group_keys:
            group_keys = sorted([k[2] for k in model_summary_map.keys() if k[0] == group_dim and k[1] == y_feature])

        for group_key in group_keys:
            try:
                model_stat = model_summary_map.get((group_dim, y_feature, group_key), {})
                fixed_drivers = key_driver_map.get((group_dim, y_feature, group_key), [])

                pos, neg = split_pos_neg(fixed_drivers)
                peer_hints = {
                    "strongest_positive_driver": pos[0]["x_feature"] if pos else None,
                    "strongest_negative_driver": neg[0]["x_feature"] if neg else None
                }

                payload = {
                    "context": {
                        "group_dim": group_dim,
                        "group_key": group_key,
                        "y_feature_raw": y_feature,
                        "y_feature": clean_feature_name(y_feature)
                    },
                    "model_stat": model_stat,
                    "fixed_drivers": fixed_drivers,
                    "peer_hints": peer_hints
                }

                payload_str = json.dumps(payload, ensure_ascii=False)

                user_prompt = build_user_prompt_detail(
                    group_dim=group_dim,
                    group_key=group_key,
                    y_feature=clean_feature_name(y_feature)
                )

                llm_json, raw = call_ai_query_with_retry_detail(
                    endpoint=ENDPOINT,
                    system_prompt=SYSTEM_PROMPT_DETAIL,
                    user_prompt=user_prompt,
                    payload_json=payload_str
                )

                llm_comments = {
                    x.get("x_feature"): x.get("comment", "")
                    for x in (llm_json.get("key_driver_comments", []) or [])
                    if x.get("x_feature")
                }

                key_drivers_final = []
                for d in fixed_drivers:
                    key_drivers_final.append({
                        "x_feature": d.get("x_feature"),
                        "coef": d.get("coef"),
                        "p_value": d.get("p_value"),
                        "direction": d.get("direction"),
                        "comment": llm_comments.get(d.get("x_feature"), "")
                    })

                detail_results.append({
                    "snapshot_quarter": SNAPSHOT_QUARTER,
                    "prompt_version": PROMPT_VERSION,
                    "group_dim": group_dim,
                    "group_key": group_key,
                    "y_feature": y_feature,
                    "y_feature_clean": clean_feature_name(y_feature),
                    "country_core": llm_json.get("country_core", ""),
                    "summary_comment": llm_json.get("summary_comment", ""),
                    "key_drivers_json": json.dumps(key_drivers_final, ensure_ascii=False),
                    "special_points_json": json.dumps(llm_json.get("special_points", []), ensure_ascii=False),
                    "raw_card_json": json.dumps(llm_json, ensure_ascii=False),
                    "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
                })

            except Exception as e:
                print(f"[ERROR] failed detail generation ({group_dim}, {group_key}, {y_feature}): {repr(e)}")
                failed_detail.append((group_dim, y_feature, group_key, repr(e)))

    print(f"[INFO] detail success={len(detail_results)}, failed={len(failed_detail)}")

    # 4. 성공 pair만 summary 생성
    detail_success_keys = {(r["group_dim"], r["y_feature"], r["group_key"]) for r in detail_results}

    successful_pairs = []
    summary_results = []

    for group_dim, y_feature in failed_pair_set:
        model_stats = pair_model_stats_map.get((group_dim, y_feature), [])
        group_keys = sorted([x["group_key"] for x in model_stats if x.get("group_key") is not None])

        if group_keys and all((group_dim, y_feature, gk) in detail_success_keys for gk in group_keys):
            successful_pairs.append((group_dim, y_feature))

            detail_seed_rows = [{"group_key": gk, "drivers": key_driver_map.get((group_dim, y_feature, gk), [])} for gk in group_keys]
            computed_confidence = compute_confidence_level(model_stats)
            y_feature_clean = clean_feature_name(y_feature)

            summary_results.append({
                "snapshot_quarter": SNAPSHOT_QUARTER,
                "prompt_version": PROMPT_VERSION,
                "group_dim": group_dim,
                "y_feature": y_feature,
                "y_feature_clean": y_feature_clean,
                "overall_core_message": make_overall_core_message(y_feature_clean, detail_seed_rows),
                "planning_summary": make_planning_summary(y_feature_clean, detail_seed_rows),
                "strategy_takeaways_json": json.dumps(make_strategy_takeaways(detail_seed_rows), ensure_ascii=False),
                "confidence_level": computed_confidence.get("level", "low"),
                "confidence_band": confidence_band_from_level(computed_confidence.get("level", "low")),
                "raw_llm_json": json.dumps({"generator": "rule_based_summary_after_detail_success"}, ensure_ascii=False),
                "payload_json": json.dumps({"context": {"group_dim": group_dim, "y_feature": y_feature}}, ensure_ascii=False),
                "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
            })

    print(f"[INFO] fully successful pairs={len(successful_pairs)}")

    # 5. detail merge
    if detail_results:
        detail_sdf = spark.createDataFrame([Row(**rec) for rec in detail_results]).select(
            F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
            F.col("prompt_version").cast("string").alias("prompt_version"),
            F.col("group_dim").cast("string").alias("group_dim"),
            F.col("group_key").cast("string").alias("group_key"),
            F.col("y_feature").cast("string").alias("y_feature"),
            F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
            F.col("country_core").cast("string").alias("country_core"),
            F.col("summary_comment").cast("string").alias("summary_comment"),
            F.col("key_drivers_json").cast("string").alias("key_drivers_json"),
            F.col("special_points_json").cast("string").alias("special_points_json"),
            F.col("raw_card_json").cast("string").alias("raw_card_json"),
            F.to_timestamp("generated_at").alias("generated_at")
        )

        detail_sdf.createOrReplaceTempView("failed_pair_detail_updates_v61")

        spark.sql(f"""
            MERGE INTO {DETAIL_TABLE} AS t
            USING failed_pair_detail_updates_v61 AS s
            ON  t.snapshot_quarter = s.snapshot_quarter
            AND t.prompt_version   = s.prompt_version
            AND t.group_dim        = s.group_dim
            AND t.group_key        = s.group_key
            AND t.y_feature        = s.y_feature
            WHEN MATCHED THEN UPDATE SET
              t.y_feature_clean     = s.y_feature_clean,
              t.country_core        = s.country_core,
              t.summary_comment     = s.summary_comment,
              t.key_drivers_json    = s.key_drivers_json,
              t.special_points_json = s.special_points_json,
              t.raw_card_json       = s.raw_card_json,
              t.generated_at        = s.generated_at
            WHEN NOT MATCHED THEN INSERT (
              snapshot_quarter,
              prompt_version,
              group_dim,
              group_key,
              y_feature,
              y_feature_clean,
              country_core,
              summary_comment,
              key_drivers_json,
              special_points_json,
              raw_card_json,
              generated_at
            )
            VALUES (
              s.snapshot_quarter,
              s.prompt_version,
              s.group_dim,
              s.group_key,
              s.y_feature,
              s.y_feature_clean,
              s.country_core,
              s.summary_comment,
              s.key_drivers_json,
              s.special_points_json,
              s.raw_card_json,
              s.generated_at
            )
        """)
        print(f"[INFO] merged detail rows: {detail_sdf.count()}")

    # 6. summary merge
    if summary_results:
        summary_sdf = spark.createDataFrame([Row(**rec) for rec in summary_results]).select(
            F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
            F.col("prompt_version").cast("string").alias("prompt_version"),
            F.col("group_dim").cast("string").alias("group_dim"),
            F.col("y_feature").cast("string").alias("y_feature"),
            F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
            F.col("overall_core_message").cast("string").alias("overall_core_message"),
            F.col("planning_summary").cast("string").alias("planning_summary"),
            F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
            F.col("confidence_level").cast("string").alias("confidence_level"),
            F.col("confidence_band").cast("string").alias("confidence_band"),
            F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
            F.col("payload_json").cast("string").alias("payload_json"),
            F.to_timestamp("generated_at").alias("generated_at")
        )

        summary_sdf.createOrReplaceTempView("failed_pair_summary_updates_v61")

        spark.sql(f"""
            MERGE INTO {SUMMARY_TABLE} AS t
            USING failed_pair_summary_updates_v61 AS s
            ON  t.snapshot_quarter = s.snapshot_quarter
            AND t.prompt_version   = s.prompt_version
            AND t.group_dim        = s.group_dim
            AND t.y_feature        = s.y_feature
            WHEN MATCHED THEN UPDATE SET
              t.y_feature_clean         = s.y_feature_clean,
              t.overall_core_message    = s.overall_core_message,
              t.planning_summary        = s.planning_summary,
              t.strategy_takeaways_json = s.strategy_takeaways_json,
              t.confidence_level        = s.confidence_level,
              t.confidence_band         = s.confidence_band,
              t.raw_llm_json            = s.raw_llm_json,
              t.payload_json            = s.payload_json,
              t.generated_at            = s.generated_at
            WHEN NOT MATCHED THEN INSERT (
              snapshot_quarter,
              prompt_version,
              group_dim,
              y_feature,
              y_feature_clean,
              overall_core_message,
              planning_summary,
              strategy_takeaways_json,
              confidence_level,
              confidence_band,
              raw_llm_json,
              payload_json,
              generated_at
            )
            VALUES (
              s.snapshot_quarter,
              s.prompt_version,
              s.group_dim,
              s.y_feature,
              s.y_feature_clean,
              s.overall_core_message,
              s.planning_summary,
              s.strategy_takeaways_json,
              s.confidence_level,
              s.confidence_band,
              s.raw_llm_json,
              s.payload_json,
              s.generated_at
            )
        """)
        print(f"[INFO] merged summary rows: {summary_sdf.count()}")

    # 7. fully successful pair만 failed 삭제
    if successful_pairs:
        delete_condition = " OR ".join([
            f"(group_dim = '{sql_escape(g)}' AND y_feature = '{sql_escape(y)}')"
            for g, y in successful_pairs
        ])

        spark.sql(f"""
            DELETE FROM {FAILED_TABLE}
            WHERE snapshot_quarter = '{sql_escape(SNAPSHOT_QUARTER)}'
              AND prompt_version = '{sql_escape(PROMPT_VERSION)}'
              AND ({delete_condition})
        """)
        print(f"[INFO] removed successful pairs from failed table: {len(successful_pairs)}")

    # 8. failed dedup
    spark.sql(f"""
    MERGE INTO {FAILED_TABLE} AS t
    USING (
      SELECT snapshot_quarter, prompt_version, group_dim, y_feature, failed_at
      FROM (
        SELECT
          snapshot_quarter, prompt_version, group_dim, y_feature, failed_at,
          ROW_NUMBER() OVER (
            PARTITION BY snapshot_quarter, prompt_version, group_dim, y_feature
            ORDER BY failed_at DESC
          ) AS rn
        FROM {FAILED_TABLE}
        WHERE snapshot_quarter = '{sql_escape(SNAPSHOT_QUARTER)}'
          AND prompt_version = '{sql_escape(PROMPT_VERSION)}'
      ) d
      WHERE rn > 1
    ) AS s
    ON  t.snapshot_quarter = s.snapshot_quarter
    AND t.prompt_version   = s.prompt_version
    AND t.group_dim        = s.group_dim
    AND t.y_feature        = s.y_feature
    AND t.failed_at        = s.failed_at
    WHEN MATCHED THEN DELETE
    """)

    print("[INFO] deduplicated failed table")

    display(
        spark.table(FAILED_TABLE)
            .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
            .filter(F.col("prompt_version") == PROMPT_VERSION)
            .orderBy(F.desc("failed_at"))
    )


In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC
import json
import re

SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION   = "v6.1.0"

MODEL_SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_model_summary"
COEF_SUMMARY_TABLE  = "sandbox.z_jungryo_lee.voc_wls_coef_summary"

SUMMARY_TABLE = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_summary_v61"
DETAIL_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_detail_v61"
FAILED_TABLE  = "sandbox.z_jungryo_lee.voc_wls_dashboard_insight_failed_v61"

PVAL_MAX = 0.10
ABS_COEF_THRESHOLD = 0.10
USE_IS_DRIVER = True
MAX_DRIVERS_PER_KEY = 3

def sql_escape(s):
    return str(s).replace("'", "''")

def safe_float(v, default=None):
    try:
        if v is None:
            return default
        return float(v)
    except Exception:
        return default

def safe_int(v, default=None):
    try:
        if v is None:
            return default
        return int(v)
    except Exception:
        return default

def clean_feature_name(name: str) -> str:
    if not name:
        return ""
    s = str(name)
    s = re.sub(r"^[0-9]+([_./-][0-9]+)*[_./-]*", "", s)
    s = re.sub(r"^[^A-Za-z가-힣]+", "", s)
    return s.replace("_", " ").strip()

def to_direction(coef):
    return "positive" if safe_float(coef, 0.0) >= 0 else "negative"

def sort_drivers(drivers):
    if not drivers:
        return []
    return sorted(drivers, key=lambda d: abs(float(d.get("coef", 0.0) or 0.0)), reverse=True)

def compute_confidence_level(model_stats: list) -> dict:
    if not model_stats:
        return {"level": "low", "score_basis": {"min_adj_r_squared": None, "max_model_p_value": None, "min_y_obs": None}}

    adj_r2_list = [safe_float(x.get("adj_r_squared")) for x in model_stats if safe_float(x.get("adj_r_squared")) is not None]
    pval_list   = [safe_float(x.get("model_p_value")) for x in model_stats if safe_float(x.get("model_p_value")) is not None]
    yobs_list   = [safe_int(x.get("y_obs")) for x in model_stats if safe_int(x.get("y_obs")) is not None]

    min_adj_r2 = min(adj_r2_list) if adj_r2_list else None
    max_pval   = max(pval_list) if pval_list else None
    min_y_obs  = min(yobs_list) if yobs_list else None

    if min_adj_r2 is None or max_pval is None or min_y_obs is None:
        level = "low"
    elif min_adj_r2 >= 0.60 and max_pval <= 0.05 and min_y_obs >= 100:
        level = "high"
    elif min_adj_r2 >= 0.30 and max_pval <= 0.10 and min_y_obs >= 60:
        level = "mid"
    else:
        level = "low"

    return {
        "level": level,
        "score_basis": {
            "min_adj_r_squared": min_adj_r2,
            "max_model_p_value": max_pval,
            "min_y_obs": min_y_obs
        }
    }

def confidence_band_from_level(level: str) -> str:
    return "Strong" if level == "high" else "Moderate" if level == "mid" else "Caution"

def split_pos_neg(drivers):
    pos = [d for d in (drivers or []) if d.get("direction") == "positive"]
    neg = [d for d in (drivers or []) if d.get("direction") == "negative"]
    pos = sorted(pos, key=lambda x: abs(float(x.get("coef", 0.0) or 0.0)), reverse=True)
    neg = sorted(neg, key=lambda x: abs(float(x.get("coef", 0.0) or 0.0)), reverse=True)
    return pos, neg

def make_driver_comment(driver):
    xf = driver.get("x_feature", "")
    if driver.get("direction") == "positive":
        return f"{xf}는 사용성 평가를 높이는 핵심 요인으로 해석됩니다."
    return f"{xf}는 사용성 평가를 저해하는 마찰 요인으로 해석됩니다."

def make_country_core(group_key, pos, neg):
    if pos and neg:
        return f"{group_key}는 {pos[0]['x_feature']}가 강점으로 작용하는 동시에 {neg[0]['x_feature']}가 제약 요인으로 나타나는 패턴입니다."
    if pos:
        return f"{group_key}는 {pos[0]['x_feature']} 중심의 가치가 두드러지는 패턴입니다."
    if neg:
        return f"{group_key}는 {neg[0]['x_feature']} 관련 마찰이 사용성 평가에 크게 작용하는 패턴입니다."
    return f"{group_key}는 뚜렷한 핵심 driver가 제한적으로 나타나는 패턴입니다."

def make_summary_comment(pos, neg):
    if pos and neg:
        return f"{pos[0]['x_feature']}의 긍정 효과가 가장 크게 나타나지만, {neg[0]['x_feature']}는 사용성 저해 요인으로 함께 확인됩니다. 따라서 강점 강화와 제약 완화가 동시에 필요한 구조입니다."
    if pos:
        return f"{pos[0]['x_feature']}가 가장 강한 긍정 driver로 나타나며, 관련 경험의 완성도를 높이는 방향이 적합합니다."
    if neg:
        return f"{neg[0]['x_feature']}가 가장 큰 음의 driver로 나타나며, 해당 영역의 불편을 줄이는 접근이 우선 과제입니다."
    return f"유의한 driver 수가 제한적이어서 특정 기능보다 전반 경험의 안정성을 보수적으로 해석할 필요가 있습니다."

def make_special_points(pos, neg):
    points = []
    if pos:
        points.append(f"{pos[0]['x_feature']}가 가장 강한 긍정 요인으로 확인됩니다.")
    if neg:
        points.append(f"{neg[0]['x_feature']}는 가장 큰 음의 요인으로 관리가 필요합니다.")
    if not points:
        points.append("뚜렷한 positive 또는 negative driver가 제한적으로 확인됩니다.")
    return points[:2]

def make_overall_core_message(y_feature_clean, detail_seed_rows):
    strongest = []
    for row in detail_seed_rows:
        pos, _ = split_pos_neg(row.get("drivers", []))
        if pos:
            strongest.append((row["group_key"], pos[0]["x_feature"], abs(float(pos[0].get("coef", 0.0) or 0.0))))
    if strongest:
        strongest = sorted(strongest, key=lambda x: x[2], reverse=True)
        g, xf, _ = strongest[0]
        return f"{y_feature_clean}는 group_key별로 핵심 driver가 다르게 나타나며, {g}에서는 {xf}가 가장 두드러진 축으로 확인됩니다."
    return f"{y_feature_clean}는 group_key별 뚜렷한 driver 차이가 제한적으로 나타나 보수적 해석이 필요한 구조입니다."

def make_planning_summary(y_feature_clean, detail_seed_rows):
    points = []
    for row in detail_seed_rows:
        pos, _ = split_pos_neg(row.get("drivers", []))
        if pos:
            points.append(f"{row['group_key']}의 {pos[0]['x_feature']}")
    if points:
        return f"{y_feature_clean}는 group_key별 강하게 작용하는 핵심 요인이 서로 다르게 나타납니다. 대표적으로 {', '.join(points[:3])}가 핵심 축으로 확인되어 공통 접근보다 group_key별 차별화 기획이 적합합니다."
    return f"{y_feature_clean}는 유의한 driver 수가 제한적이어서 특정 기능 축보다 전반 경험 안정성 중심으로 해석하는 편이 적절합니다."

def make_strategy_takeaways(detail_seed_rows):
    takeaways = []
    positives, negatives = [], []

    for row in detail_seed_rows:
        pos, neg = split_pos_neg(row.get("drivers", []))
        if pos:
            positives.append((row["group_key"], pos[0]))
        if neg:
            negatives.append((row["group_key"], neg[0]))

    if positives:
        g, d = sorted(positives, key=lambda x: abs(float(x[1].get("coef", 0.0) or 0.0)), reverse=True)[0]
        takeaways.append(f"{g}는 {d['x_feature']} 중심의 강점 강화가 우선 과제입니다.")
    if negatives:
        g, d = sorted(negatives, key=lambda x: abs(float(x[1].get("coef", 0.0) or 0.0)), reverse=True)[0]
        takeaways.append(f"{g}는 {d['x_feature']} 관련 마찰을 우선 완화할 필요가 있습니다.")
    if len(detail_seed_rows) >= 2:
        takeaways.append("group_key별 핵심 가치 축이 다르므로 단일 메시지보다 세분화된 기획 전략이 적합합니다.")
    if not takeaways:
        takeaways.append("유의한 driver가 제한적이므로 보수적으로 해석할 필요가 있습니다.")
    return takeaways[:3]

# 1) 마지막 실패 1건 읽기
last_failed = (
    spark.table(FAILED_TABLE)
    .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
    .filter(F.col("prompt_version") == PROMPT_VERSION)
    .orderBy(F.desc("failed_at"))
    .limit(1)
    .collect()
)

if not last_failed:
    print("[INFO] no failed rows")
else:
    failed_row = last_failed[0]
    group_dim = failed_row["group_dim"]
    y_feature = failed_row["y_feature"]

    print(f"[INFO] fallback target: group_dim={group_dim}, y_feature={y_feature}")

    # 2) 원천 데이터 재구성
    model_summary_rows = (
        spark.table(MODEL_SUMMARY_TABLE)
        .filter(F.col("group_dim") == group_dim)
        .filter(F.col("y_feature") == y_feature)
        .collect()
    )

    coef_rows = (
        spark.table(COEF_SUMMARY_TABLE)
        .filter(F.col("group_dim") == group_dim)
        .filter(F.col("y_feature") == y_feature)
        .collect()
    )

    model_stats = []
    group_keys = []

    for r in model_summary_rows:
        model_stats.append({
            "group_key": r["group_key"],
            "r_squared": r["r_squared"],
            "adj_r_squared": r["adj_r_squared"],
            "model_p_value": r["prob_f"],
            "y_obs": r["y_obs"]
        })
        group_keys.append(r["group_key"])

    group_keys = sorted(list(set(group_keys)))

    key_driver_map = {}
    for r in coef_rows:
        coef = r["coef"]
        p_value = r["p_value"]
        abs_coef = abs(float(coef or 0.0))
        is_driver_ok = True
        if USE_IS_DRIVER and "is_driver" in r.asDict():
            is_driver_ok = (r["is_driver"] == 1)

        if is_driver_ok and p_value is not None and p_value <= PVAL_MAX and abs_coef >= ABS_COEF_THRESHOLD:
            k = (r["group_key"])
            key_driver_map.setdefault(k, []).append({
                "x_feature": clean_feature_name(r["x_feature"]),
                "coef": coef,
                "p_value": p_value,
                "direction": to_direction(coef)
            })

    for k in list(key_driver_map.keys()):
        key_driver_map[k] = sort_drivers(key_driver_map[k])[:MAX_DRIVERS_PER_KEY]

    # 3) detail rule-based 생성
    detail_results = []
    detail_seed_rows = []

    for group_key in group_keys:
        drivers = key_driver_map.get(group_key, [])
        pos, neg = split_pos_neg(drivers)

        key_drivers_final = []
        for d in drivers:
            key_drivers_final.append({
                "x_feature": d.get("x_feature"),
                "coef": d.get("coef"),
                "p_value": d.get("p_value"),
                "direction": d.get("direction"),
                "comment": make_driver_comment(d)
            })

        detail_results.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "group_key": group_key,
            "y_feature": y_feature,
            "y_feature_clean": clean_feature_name(y_feature),
            "country_core": make_country_core(group_key, pos, neg),
            "summary_comment": make_summary_comment(pos, neg),
            "key_drivers_json": json.dumps(key_drivers_final, ensure_ascii=False),
            "special_points_json": json.dumps(make_special_points(pos, neg), ensure_ascii=False),
            "raw_card_json": json.dumps({"generator": "rule_based_final_fallback"}, ensure_ascii=False),
            "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
        })

        detail_seed_rows.append({
            "group_key": group_key,
            "drivers": drivers
        })

    # 4) summary rule-based 생성
    computed_confidence = compute_confidence_level(model_stats)

    summary_result = {
        "snapshot_quarter": SNAPSHOT_QUARTER,
        "prompt_version": PROMPT_VERSION,
        "group_dim": group_dim,
        "y_feature": y_feature,
        "y_feature_clean": clean_feature_name(y_feature),
        "overall_core_message": make_overall_core_message(clean_feature_name(y_feature), detail_seed_rows),
        "planning_summary": make_planning_summary(clean_feature_name(y_feature), detail_seed_rows),
        "strategy_takeaways_json": json.dumps(make_strategy_takeaways(detail_seed_rows), ensure_ascii=False),
        "confidence_level": computed_confidence.get("level", "low"),
        "confidence_band": confidence_band_from_level(computed_confidence.get("level", "low")),
        "raw_llm_json": json.dumps({"generator": "rule_based_final_fallback"}, ensure_ascii=False),
        "payload_json": json.dumps({"context": {"group_dim": group_dim, "y_feature": y_feature}}, ensure_ascii=False),
        "generated_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
    }

    # 5) summary merge
    summary_sdf = spark.createDataFrame([Row(**summary_result)]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("overall_core_message").cast("string").alias("overall_core_message"),
        F.col("planning_summary").cast("string").alias("planning_summary"),
        F.col("strategy_takeaways_json").cast("string").alias("strategy_takeaways_json"),
        F.col("confidence_level").cast("string").alias("confidence_level"),
        F.col("confidence_band").cast("string").alias("confidence_band"),
        F.col("raw_llm_json").cast("string").alias("raw_llm_json"),
        F.col("payload_json").cast("string").alias("payload_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    summary_sdf.createOrReplaceTempView("last_failed_summary_fallback")

    spark.sql(f"""
        MERGE INTO {SUMMARY_TABLE} AS t
        USING last_failed_summary_fallback AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean         = s.y_feature_clean,
          t.overall_core_message    = s.overall_core_message,
          t.planning_summary        = s.planning_summary,
          t.strategy_takeaways_json = s.strategy_takeaways_json,
          t.confidence_level        = s.confidence_level,
          t.confidence_band         = s.confidence_band,
          t.raw_llm_json            = s.raw_llm_json,
          t.payload_json            = s.payload_json,
          t.generated_at            = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter,
          prompt_version,
          group_dim,
          y_feature,
          y_feature_clean,
          overall_core_message,
          planning_summary,
          strategy_takeaways_json,
          confidence_level,
          confidence_band,
          raw_llm_json,
          payload_json,
          generated_at
        )
        VALUES (
          s.snapshot_quarter,
          s.prompt_version,
          s.group_dim,
          s.y_feature,
          s.y_feature_clean,
          s.overall_core_message,
          s.planning_summary,
          s.strategy_takeaways_json,
          s.confidence_level,
          s.confidence_band,
          s.raw_llm_json,
          s.payload_json,
          s.generated_at
        )
    """)

    # 6) detail merge
    detail_sdf = spark.createDataFrame([Row(**rec) for rec in detail_results]).select(
        F.col("snapshot_quarter").cast("string").alias("snapshot_quarter"),
        F.col("prompt_version").cast("string").alias("prompt_version"),
        F.col("group_dim").cast("string").alias("group_dim"),
        F.col("group_key").cast("string").alias("group_key"),
        F.col("y_feature").cast("string").alias("y_feature"),
        F.col("y_feature_clean").cast("string").alias("y_feature_clean"),
        F.col("country_core").cast("string").alias("country_core"),
        F.col("summary_comment").cast("string").alias("summary_comment"),
        F.col("key_drivers_json").cast("string").alias("key_drivers_json"),
        F.col("special_points_json").cast("string").alias("special_points_json"),
        F.col("raw_card_json").cast("string").alias("raw_card_json"),
        F.to_timestamp("generated_at").alias("generated_at")
    )

    detail_sdf.createOrReplaceTempView("last_failed_detail_fallback")

    spark.sql(f"""
        MERGE INTO {DETAIL_TABLE} AS t
        USING last_failed_detail_fallback AS s
        ON  t.snapshot_quarter = s.snapshot_quarter
        AND t.prompt_version   = s.prompt_version
        AND t.group_dim        = s.group_dim
        AND t.group_key        = s.group_key
        AND t.y_feature        = s.y_feature
        WHEN MATCHED THEN UPDATE SET
          t.y_feature_clean     = s.y_feature_clean,
          t.country_core        = s.country_core,
          t.summary_comment     = s.summary_comment,
          t.key_drivers_json    = s.key_drivers_json,
          t.special_points_json = s.special_points_json,
          t.raw_card_json       = s.raw_card_json,
          t.generated_at        = s.generated_at
        WHEN NOT MATCHED THEN INSERT (
          snapshot_quarter,
          prompt_version,
          group_dim,
          group_key,
          y_feature,
          y_feature_clean,
          country_core,
          summary_comment,
          key_drivers_json,
          special_points_json,
          raw_card_json,
          generated_at
        )
        VALUES (
          s.snapshot_quarter,
          s.prompt_version,
          s.group_dim,
          s.group_key,
          s.y_feature,
          s.y_feature_clean,
          s.country_core,
          s.summary_comment,
          s.key_drivers_json,
          s.special_points_json,
          s.raw_card_json,
          s.generated_at
        )
    """)

    # 7) 성공했으니 failed 삭제
    spark.sql(f"""
        DELETE FROM {FAILED_TABLE}
        WHERE snapshot_quarter = '{sql_escape(SNAPSHOT_QUARTER)}'
          AND prompt_version = '{sql_escape(PROMPT_VERSION)}'
          AND group_dim = '{sql_escape(group_dim)}'
          AND y_feature = '{sql_escape(y_feature)}'
    """)

    print("[INFO] last failed row resolved by rule-based fallback and removed from failed table")

    display(
        spark.table(FAILED_TABLE)
            .filter(F.col("snapshot_quarter") == SNAPSHOT_QUARTER)
            .filter(F.col("prompt_version") == PROMPT_VERSION)
            .orderBy(F.desc("failed_at"))
    )
